<a href="https://colab.research.google.com/github/IvanMorsin/Forecasting-electrical-power-in-multi-storey-residential-buildings/blob/main/notebook_14_economics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaleido==0.2.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 10.0 MB/s eta 0:00:00


In [ ]:
!pip install workalendar -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 48.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.4 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Трансформаторы (источник: tszi.ru, 2024-2025)
COST_TMG_2000 = 2_060_000
N_TRANSFORMERS = 2

# Корпус КТП 2000 кВА (источник: innoka-group.ru, 2024-2025)
COST_KTP_ENCLOSURE = 1_760_000

COST_CABLE_10KV_PER_M = 5_000
COST_CABLE_04KV_PER_M = 2_000
CABLE_10KV_LENGTH = 2500
CABLE_04KV_LENGTH = 250

COST_INSTALLATION_RATE = 0.30


COST_DESIGN_TP = 2_000_000  # руб


cost_transformers = COST_TMG_2000 * N_TRANSFORMERS
cost_ktp = COST_KTP_ENCLOSURE
cost_equipment = cost_transformers + cost_ktp
cost_smr = cost_equipment * COST_INSTALLATION_RATE
cost_cable_10kv = COST_CABLE_10KV_PER_M * CABLE_10KV_LENGTH
cost_cable_04kv = COST_CABLE_04KV_PER_M * CABLE_04KV_LENGTH
cost_design = COST_DESIGN_TP

cost_scenario_A = (cost_equipment + cost_smr +
                   cost_cable_10kv + cost_cable_04kv +
                   cost_design)

items_A = {
    f'Трансформаторы\n2×ТМГ 2000 кВА':  cost_transformers,
    'Корпус КТП\nс ячейками ВН/НН': cost_ktp,
    f'СМР\n(30% от оборудования)': cost_smr,
    f'КЛ 10 кВ\n({CABLE_10KV_LENGTH} м)': cost_cable_10kv,
    f'КЛ 0.4 кВ\n({CABLE_04KV_LENGTH} м)': cost_cable_04kv,
    'Проектирование\nи техприсоединение': cost_design,
}
print("СЦЕНАРИЙ А - Строительство новой ТП 2×ТМГ 2000 кВА")
for name, cost in items_A.items():
    clean = name.replace('\n', ' ')
    print(f"  {clean:<45} {cost:>12,.0f} руб")

print(f"  {'ИТОГО Сценарий А':<45} {cost_scenario_A:>12,.0f} руб")
print()
print("СЦЕНАРИЙ Б - Использование резерва существующих ВРУ")
print(f"  {'Капитальные затраты':<45} {'0':>12} руб")
print(f"  (подключение к существующим ВРУ домов)")

print()
print(f"  ЭКОНОМИЯ (А - Б): {cost_scenario_A:,.0f} руб "
      f"= {cost_scenario_A/1e6:.2f} млн руб")

# Таблица
df_econ = pd.DataFrame([
    {'Статья затрат': k.replace('\n', ' '), 'Сценарий А (руб)': v, 'Сценарий Б (руб)': 0}
    for k, v in items_A.items()
])
df_econ.loc[len(df_econ)] = {
    'Статья затрат': 'ИТОГО',
    'Сценарий А (руб)': cost_scenario_A,
    'Сценарий Б (руб)': 0
}
df_econ['Экономия (руб)'] = df_econ['Сценарий А (руб)'] - df_econ['Сценарий Б (руб)']
print()
print(df_econ.to_string(index=False))
df_econ.to_csv('results_economic.csv', index=False)

СЦЕНАРИЙ А - Строительство новой ТП 2×ТМГ 2000 кВА
  Трансформаторы 2×ТМГ 2000 кВА                    4,120,000 руб
  Корпус КТП с ячейками ВН/НН                      1,760,000 руб
  СМР (30% от оборудования)                        1,764,000 руб
  КЛ 10 кВ (2500 м)                               12,500,000 руб
  КЛ 0.4 кВ (250 м)                                  500,000 руб
  Проектирование и техприсоединение                2,000,000 руб
  ИТОГО Сценарий А                                22,644,000 руб

СЦЕНАРИЙ Б - Использование резерва существующих ВРУ
  Капитальные затраты                                      0 руб
  (подключение к существующим ВРУ домов)

  ЭКОНОМИЯ (А - Б): 22,644,000 руб = 22.64 млн руб

                    Статья затрат  Сценарий А (руб)  Сценарий Б (руб)  Экономия (руб)
    Трансформаторы 2×ТМГ 2000 кВА         4120000.0                 0       4120000.0
      Корпус КТП с ячейками ВН/НН         1760000.0                 0       1760000.0
        СМР (30% от обор

In [ ]:
#График 1: сравнение А и Б
labels = [k.replace('\n', '<br>') for k in items_A.keys()]
values = list(items_A.values())
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

fig1 = go.Figure(go.Bar(
    x=['Сценарий А<br>(новая ТП 2×ТМГ 2000)', 'Сценарий Б<br>(резерв ВРУ)'],
    y=[cost_scenario_A / 1e6, 0],
    marker_color=['#d62728', '#2ca02c'],
    text=[f'{cost_scenario_A/1e6:.2f} млн руб', '0 руб'],
    textposition='outside',
    width=0.4
))

fig1.update_layout(
    title=f'Сравнение сценариев подключения зарядных станций<br>'
          f'Экономия при выборе Сценария Б: {cost_scenario_A/1e6:.2f} млн руб',
    yaxis_title='Капитальные затраты, млн руб',
    template='plotly_white',
    height=450, width=600,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=70, b=60),
    showlegend=False
)
fig1.show()
fig1.write_image('nb16_scenario_AB_compare.png', scale=2)

#График 2: водопадная диаграмма
measure = ['relative'] * len(items_A) + ['total']
text_vals = [f'{v/1e6:.2f} млн' for v in values] + [f'{cost_scenario_A/1e6:.2f} млн']

fig2 = go.Figure(go.Waterfall(
    orientation='v',
    measure=measure,
    x=labels + ['ИТОГО'],
    y=[v / 1e6 for v in values] + [0],
    text=text_vals,
    textposition='outside',
    connector=dict(line=dict(color='grey', width=1)),
    increasing=dict(marker=dict(color='#1f77b4')),
    totals=dict(marker=dict(color='#d62728'))
))

fig2.update_layout(
    title='Водопадная диаграмма: формирование затрат Сценария А',
    yaxis_title='Стоимость, млн руб',
    template='plotly_white',
    height=450, width=800,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=60, b=80),
)
fig2.show()
fig2.write_image('nb16_waterfall.png', scale=2)